# RAG using Langchain

## Packages loading & import

In [81]:
# !pip install "langchain-core>=0.2.0,<0.3.0" \
#              "langchain>=0.2.0,<0.3.0" \
#              "langchain-community>=0.2.0,<0.3.0" \
#              "langchain-huggingface>=0.0.3,<0.1.0" \
#              "langchain-chroma>=0.1.0,<0.2.0" \
#              "langchain-ollama>=0.1.0,<0.2.0" \
#              "langchain-text-splitters>=0.2.0,<0.3.0" \
#              "transformers>=4.39.0" \
#              "accelerate>=0.28.0" \
#              "sentence-transformers" \
#              rank-bm25 \
#              huggingface_hub \
#              tqdm \
#              beautifulsoup4

In [82]:
import os
from dotenv import load_dotenv

load_dotenv()

# ... rest of the cell ...
import json
import bs4
import nltk
import torch
import pickle
import numpy as np

# from pyserini.index import IndexWriter
# from pyserini.search import SimpleSearcher
from numpy.linalg import norm
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize

from langchain_community.llms import Ollama
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain.vectorstores import Chroma
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.embeddings import JinaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain.docstore.document import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

from tqdm import tqdm

In [83]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/eason/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/eason/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Hugging face login
- Please apply the model first: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
- If you haven't been granted access to this model, you can use other LLM model that doesn't have to apply.
- You must save the hf token otherwise you need to regenrate the token everytime.
- When using Ollama, no login is required to access and utilize the llama model.

In [84]:
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN environment variable not set. Please set it in .env file.")
login(token=hf_token, add_to_git_credential=True)

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


In [85]:
!huggingface-cli whoami

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


⚠️  Warning: 'huggingface-cli whoami' is deprecated. Use 'hf auth whoami' instead.
Eason100


## TODO1: Set up the environment of Ollama

### Introduction to Ollama
- Ollama is a platform designed for running and managing large language models (LLMs) directly **on local devices**, providing a balance between performance, privacy, and control.
- There are also other tools support users to manage LLM on local devices and accelerate it like *vllm*, *Llamafile*, *GPT4ALL*...etc.

### Launch colabxterm

In [86]:
# TODO1-1: You should install colab-xterm and launch it.
# Write your commands here.

In [87]:
# TODO1-2: You should install Ollama.
# You may need root privileges if you use a local machine instead of Colab.

In [88]:
# %xterm

In [89]:
# TODO1-3: Pull Llama3.2:1b via Ollama and start the Ollama service in the xterm
# Write your commands in the xterm

## Ollama testing
You can test your Ollama status with the following cells.

In [90]:
# Setting up the model that this tutorial will use
MODEL = "llama3.2:1b" # https://ollama.com/library/llama3.2:3b
EMBED_MODEL = "jinaai/jina-embeddings-v2-base-en"

In [91]:
# Initialize an instance of the Ollama model
llm = Ollama(model=MODEL)
# Invoke the model to generate responses
response = llm.invoke("What is the capital of Taiwan?")
print(response)

The capital of Taiwan is Taipei.


## Build a simple RAG system by using LangChain

### TODO2: Load the cat-facts dataset and prepare the retrieval database

In [92]:
!wget https://huggingface.co/ngxson/demo_simple_rag_py/resolve/main/cat-facts.txt

--2025-12-08 01:38:57--  https://huggingface.co/ngxson/demo_simple_rag_py/resolve/main/cat-facts.txt
Resolving huggingface.co (huggingface.co)... 2600:9000:284f:2e00:17:b174:6d00:93a1, 2600:9000:284f:8c00:17:b174:6d00:93a1, 2600:9000:284f:800:17:b174:6d00:93a1, ...
Connecting to huggingface.co (huggingface.co)|2600:9000:284f:2e00:17:b174:6d00:93a1|:443... connected.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/ngxson/demo_simple_rag_py/ccd6b7b72b52c7ca4e8f2a0a00b15c368d6ae294/cat-facts.txt?%2Fngxson%2Fdemo_simple_rag_py%2Fresolve%2Fmain%2Fcat-facts.txt=&etag=%22bc94ddd9483183e01bcf61e8bf9450fe3e09edb3%22 [following]
--2025-12-08 01:38:57--  https://huggingface.co/api/resolve-cache/models/ngxson/demo_simple_rag_py/ccd6b7b72b52c7ca4e8f2a0a00b15c368d6ae294/cat-facts.txt?%2Fngxson%2Fdemo_simple_rag_py%2Fresolve%2Fmain%2Fcat-facts.txt=&etag=%22bc94ddd9483183e01bcf61e8bf9450fe3e09edb3%22
Reusing existing connection to [huggingface.co]:443.
HTTP request sent, awaiting response... 200 OK
Length: 22657 (22K) [text/plain]
Saving to: ‘cat-facts.txt.19’

cat-facts.txt.19    100%[===================>]  22.13K  --.-KB/s    in 0s      

2025-12-08 01:38:58 (151 MB/s) - ‘cat-facts.txt.19’ saved [22657/22657]



In [93]:
# TODO2-1: Load the cat-facts dataset (as `refs`, which is a list of strings for all the cat facts)
with open('cat-facts.txt', 'r') as f:
    refs = [line.strip() for line in f if line.strip()]


In [ ]:
from langchain_core.documents import Document

docs = [Document(page_content=doc, metadata={"id": i}) for i, doc in enumerate(refs)]

# docs = [Document(page_content=f"Fact {i+1}: {doc}", metadata={"id": i}) for i, doc in enumerate(refs)]

# docs = [Document(page_content=doc.lower(), metadata={"id": i}) for i, doc in enumerate(refs)]

# docs = [Document(page_content=f"[CAT] {doc}", metadata={"id": i}) for i, doc in enumerate(refs)]

print(f"Using format: Raw text (original)")
print(f"Example: {docs[0].page_content[:80]}...")

Using format: [CAT] tag
Example: [CAT] On average, cats spend 2/3 of every day sleeping. That means a nine-year-o...


In [95]:
# Create an embedding model
model_kwargs = {'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': False}
embeddings_model = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

In [96]:
# TODO2-2: Prepare the retrieval database
# You should create a Chroma vector store.
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings_model,
    collection_name="cat_facts"
)
# We set k=5 here to ensure we can measure Recall@5
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)


### Prompt setting

In [97]:
# TODO3: Set up the `system_prompt` and configure the prompt.
system_prompt = (
    "Read the context and answer the question in 1-5 words.\n"
    "Be precise. Only output the answer, nothing else.\n\n"
    "Follow the instructions, else I'll punish you."
    "Context:\n{context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "Question: {input}\nAnswer:"),
    ]
)


- For the vectorspace, the common algorithm would be used like Faiss, Chroma...(https://python.langchain.com/docs/integrations/vectorstores/) to deal with the extreme huge database.

In [98]:
# TODO4: Build and run the RAG system
# TODO4-1: Load the QA chain
# You should create a chain for passing a list of Documents to a model.
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# TODO4-2: Create retrieval chain
# You should create retrieval chain that retrieves documents and then passes them on.
chain = create_retrieval_chain(retriever, question_answer_chain)


In [99]:
# Question (queries) and answer pairs
# Please load the questions_answers.txt file and prepare the `queries` and `answers` lists.
with open('questions_answers.txt', 'r') as f:
    # Filter out blank lines first!
    lines = [l.strip() for l in f if l.strip()]

# Now lines should be: Q1, A1, Q2, A2, Q3, A3, ... (300 lines for 150 pairs)
queries = lines[0::2]  # Every even index: Q1, Q2, Q3...
answers = lines[1::2]  # Every odd index: A1, A2, A3...

print(f"Loaded {len(queries)} queries and {len(answers)} answers.")


Loaded 150 queries and 150 answers.


In [100]:
results = []
correct_count = 0
recall_1_count = 0
recall_5_count = 0

print("Starting evaluation...")

for i, query in tqdm(enumerate(queries), total=len(queries)):
    retrieved_docs = retriever.get_relevant_documents(query)
    
    target_id = i
    
    # Calculate Recall@1
    if len(retrieved_docs) > 0 and retrieved_docs[0].metadata.get('id') == target_id:
        recall_1_count += 1
        
    found_in_top_5 = any(doc.metadata.get('id') == target_id for doc in retrieved_docs[:5])
    if found_in_top_5:
        recall_5_count += 1

    # 2. Generate Answer
    # We pass the retrieved docs to the chain
    response = question_answer_chain.invoke({"input": query, "context": retrieved_docs})

    pred = response
    if isinstance(response, dict) and 'answer' in response:
        pred = response['answer']
    

    is_correct = answers[i].lower() in pred.lower()
    if is_correct:
        correct_count += 1
    
    results.append({
        "Query": query,
        "Ground_Truth": answers[i],
        "Prediction": pred
    })

total = len(queries)
print(f"Recall@1: {recall_1_count / total:.4f}")
print(f"Recall@5: {recall_5_count / total:.4f}")
print(f"EM Accuracy: {correct_count / total:.4f}")


with open('NLP_HW4_NTHU_112062223.json', 'w') as f:
    json.dump(results, f, indent=4)


Starting evaluation...


100%|██████████| 150/150 [00:28<00:00,  5.34it/s]

Recall@1: 0.9267
Recall@5: 0.9733
EM Accuracy: 0.4600
